In [1]:
import numpy as np
import pandas as pd 
from datasets import load_dataset

In [2]:
ds = load_dataset("OpenAssistant/oasst1")
train = ds['train'].to_pandas()
val = ds['validation'].to_pandas()

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Dataset parquet downloaded and prepared to /root/.cache/huggingface/datasets/parquet/OpenAssistant--oasst1-7b815f0e66cb7b52/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901. Subsequent calls will reuse this data.


  0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
def prep_data(df):
    df_assistant = df[(df.role == "assistant") & (df["rank"] == 0.0)].copy()
    df_prompter = df[(df.role == "prompter")].copy()
    df_prompter = df_prompter.set_index("message_id")
    df_assistant["output"] = df_assistant["text"].values

    inputs = []
    parent_ids = []
    for _, row in df_assistant.iterrows():
        input = df_prompter.loc[row.parent_id]
        inputs.append(input.text)
        parent_ids.append(input.parent_id)

    df_assistant["instruction"] = inputs
    df_assistant["parent_id"] = parent_ids

    df_assistant = df_assistant[df_assistant.lang == "en"]

    df_assistant = df_assistant[
        ["instruction", "output", "message_id", "parent_id"]
    ].rename(columns={"message_id": "id"})
    
    return df_assistant

In [4]:
df_train = prep_data(train)
df_val = prep_data(val)

In [5]:
len(df_train), len(df_val)

(7856, 418)

In [6]:
pd.concat([df_train, df_val]).reset_index(drop=True).to_csv("train_full.csv", index=False)

In [7]:
df_train.to_csv("train.csv", index=False)
df_val.to_csv("val.csv", index=False)